# Pós-processamento QKD: AIT-QKD vs. qkdpp

Pipeline automatizado: leitura do CSV bruto de bancada -> sifting -> PE -> EC -> confirmação -> PA,
rodado em paralelo pelo `ait-qkd` (C++, via módulos) e pelo `qkdpp` (Python).

**Antes de rodar**: coloque `rondas_completas.csv` em `/work/run1/` (você já indicou que está lá).
Rode as células em ordem, de cima para baixo, sem pular nenhuma.

In [ ]:
import sys, time, importlib
from pathlib import Path

sys.path.insert(0, "/work")
import qkdrun as q
importlib.reload(q)

work = Path("/work/run1")
work.mkdir(exist_ok=True, parents=True)

q.ensure_dbus()

## 1. Limpeza
Mata qualquer processo/estado de rodadas anteriores. Rode sempre antes de uma nova rodada completa.

In [ ]:
import subprocess, shutil

subprocess.run(["pkill", "-9", "-f", "/opt/qkd/build/bin/qkd-"], check=False)
subprocess.run(["pkill", "-9", "-f", "dbus-daemon --config-file"], check=False)
time.sleep(2)

shutil.rmtree("/tmp/qkd-test-dbus", ignore_errors=True)
shutil.rmtree("/tmp/qkd", ignore_errors=True)
for f in work.glob("*.log"):
    f.unlink()

q.ensure_dbus()
print("bus limpo, módulos registrados agora:", q.module_count())

## 2. Leitura do CSV e sifting

Regra de sifting usada, validada contra os dados que você já tinha nos passado (bate 100%):

- `base_match == 1` (Alice e Bob escolheram a mesma base)
- `detectada == 1` (houve clique)
- `detector != 'ambos'` (exclui cliques duplos/ambíguos)

Bit de Alice = `bitA`. Bit de Bob = `bitA XOR error` (a coluna `error` já é, por definição, a discordância entre a chave de Alice e o resultado de Bob nesse round).

Se você usar isso com um CSV de formato diferente, as asserções abaixo vão falhar de forma clara em vez de produzir silenciosamente uma chave errada -- ajuste a regra de sifting para o seu esquema de colunas.

In [ ]:
import pandas as pd
import numpy as np

csv_path = work / "rondas_completas.csv"
assert csv_path.exists(), f"não encontrei {csv_path} -- confira o nome/local do arquivo"

raw = pd.read_csv(csv_path)
print(f"linhas no CSV bruto: {len(raw)}")

required_cols = {"round_id", "base_match", "detectada", "detector", "bitA", "error"}
missing = required_cols - set(raw.columns)
assert not missing, f"colunas esperadas ausentes: {missing}"

mask = (raw.base_match == 1) & (raw.detectada == 1) & (raw.detector != "ambos")
sifted = raw[mask].sort_values("round_id")

assert sifted["error"].notna().all(), "sifting selecionou linhas sem taxa de erro definida -- regra errada para este CSV"

alice_bits = sifted["bitA"].astype(int).to_numpy()
bob_bits = (sifted["bitA"].astype(int) ^ sifted["error"].astype(int)).to_numpy()

n_sifted = len(alice_bits)
qber_true = float(np.mean(alice_bits != bob_bits))
print(f"bits siftados: {n_sifted}")
print(f"QBER verdadeiro (diagnóstico, não use como e_ph de produção): {qber_true:.4f}")

alice_txt = work / "alice_sifted.txt"
bob_txt = work / "bob_sifted.txt"
alice_txt.write_text("".join(map(str, alice_bits.tolist())))
bob_txt.write_text("".join(map(str, bob_bits.tolist())))
print(f"salvos: {alice_txt.name}, {bob_txt.name}")

## 3. Pós-processamento pelo `ait-qkd`

`enkey -> error-estimation (PE) -> cascade (EC) -> confirmation -> privacy-amplification (PA) -> dekey`

Notas de implementação (bugs que já caçamos nesta conversa, corrigidos aqui):
- `enkey.key_size` é uma chave **sem** prefixo de papel (compartilhada entre alice/bob).
- `file_url` precisa ser `file://<caminho absoluto>` -- um nome relativo é rejeitado pelo `QUrl::isLocalFile()`.
- `terminate_after` só é setado no **último** módulo (`dekey`) -- setar em todos causa perda de mensagens ZeroMQ na race de encerramento.

In [ ]:
CHUNK_BYTES = 128    # tamanho de cada "chave" processada pelo Cascade
PE_DISCLOSE = 0.1    # fração revelada publicamente para estimar QBER (mesma convenção do qkdpp)
PA_SECURITY_BITS = 100   # margem de segurança fixa da PA (AIT); ~equivalente ao pa_cost do qkdpp

a_bits = alice_bits.astype(np.uint8)
b_bits = bob_bits.astype(np.uint8)

n_bytes = (n_sifted // 8 // CHUNK_BYTES) * CHUNK_BYTES
n_bits_used = n_bytes * 8
n_chunks = n_bytes // CHUNK_BYTES
a_bits_ait, b_bits_ait = a_bits[:n_bits_used], b_bits[:n_bits_used]
print(f"AIT: usando {n_bits_used} de {n_sifted} bits siftados ({n_chunks} chunks de {CHUNK_BYTES}B; "
      f"{n_sifted - n_bits_used} bits de cauda descartados por não fecharem um chunk exato)")

alice_bin = work / "alice_packed.bin"
bob_bin = work / "bob_packed.bin"
np.packbits(a_bits_ait).tofile(alice_bin)
np.packbits(b_bits_ait).tofile(bob_bin)

ait_config = f"""[module]

enkey.key_size = {CHUNK_BYTES}
enkey.alice.file_url = file://{alice_bin.resolve()}
enkey.alice.url_pipe_out = ipc:///tmp/qkd/ee.alice.in
enkey.bob.file_url = file://{bob_bin.resolve()}
enkey.bob.url_pipe_out = ipc:///tmp/qkd/ee.bob.in

error-estimation.alice.url_peer = tcp://127.0.0.1:7140
error-estimation.alice.url_pipe_in = ipc:///tmp/qkd/ee.alice.in
error-estimation.alice.url_pipe_out = ipc:///tmp/qkd/cascade.alice.in
error-estimation.bob.url_listen = tcp://127.0.0.1:7140
error-estimation.bob.url_pipe_in = ipc:///tmp/qkd/ee.bob.in
error-estimation.bob.url_pipe_out = ipc:///tmp/qkd/cascade.bob.in
error-estimation.disclose = {PE_DISCLOSE}

cascade.alice.url_peer = tcp://127.0.0.1:7130
cascade.alice.url_pipe_in = ipc:///tmp/qkd/cascade.alice.in
cascade.alice.url_pipe_out = ipc:///tmp/qkd/confirmation.alice.in
cascade.bob.url_listen = tcp://127.0.0.1:7130
cascade.bob.url_pipe_in = ipc:///tmp/qkd/cascade.bob.in
cascade.bob.url_pipe_out = ipc:///tmp/qkd/confirmation.bob.in
cascade.passes = 14

confirmation.alice.url_peer = tcp://127.0.0.1:7160
confirmation.alice.url_pipe_in = ipc:///tmp/qkd/confirmation.alice.in
confirmation.alice.url_pipe_out = ipc:///tmp/qkd/pa.alice.in
confirmation.bob.url_listen = tcp://127.0.0.1:7160
confirmation.bob.url_pipe_in = ipc:///tmp/qkd/confirmation.bob.in
confirmation.bob.url_pipe_out = ipc:///tmp/qkd/pa.bob.in
confirmation.rounds = 10

privacy-amplification.alice.url_peer = tcp://127.0.0.1:7180
privacy-amplification.alice.url_pipe_in = ipc:///tmp/qkd/pa.alice.in
privacy-amplification.alice.url_pipe_out = ipc:///tmp/qkd/dekey.alice.in
privacy-amplification.bob.url_listen = tcp://127.0.0.1:7180
privacy-amplification.bob.url_pipe_in = ipc:///tmp/qkd/pa.bob.in
privacy-amplification.bob.url_pipe_out = ipc:///tmp/qkd/dekey.bob.in
privacy-amplification.security_bits = {PA_SECURITY_BITS}

dekey.terminate_after = {n_chunks}
dekey.alice.url_pipe_in = ipc:///tmp/qkd/dekey.alice.in
dekey.alice.file_url = file://{(work / 'alice_final_ait.bin').resolve()}
dekey.bob.url_pipe_in = ipc:///tmp/qkd/dekey.bob.in
dekey.bob.file_url = file://{(work / 'bob_final_ait.bin').resolve()}
"""
config_ait = work / "pipeline_ait.conf"
config_ait.write_text(ait_config)

procs = []
for module in ["dekey", "privacy-amplification", "confirmation", "cascade", "error-estimation", "enkey"]:
    procs += q.launch(module, config_ait, work)

ok = q.wait_for_modules(12, timeout=30)
print("módulos:", q.module_count(), "| ok:", ok)

In [ ]:
for i in range(60):
    time.sleep(1)
    n_registered = q.module_count()
    if i % 5 == 0:
        print(f"t={i}s  módulos ativos: {n_registered}")
    p = work / "bob_final_ait.bin"
    if p.exists() and p.stat().st_size > 0:
        print("dekey já escreveu e fechou o arquivo")
        time.sleep(2)
        break

q.kill_all(procs)

print("\n--- arquivos finais AIT ---")
for name in ["alice_final_ait.bin", "bob_final_ait.bin"]:
    p = work / name
    print(name, p.stat().st_size if p.exists() else "NÃO EXISTE", "bytes")

## 4. Pós-processamento pelo `qkdpp`

`sifting.estimate_qber -> cascade.reconcile -> extract.verify -> extract.amplify`, tudo via `qkdpp.run()` (já inclui os patches de segurança: schedule sem colapso de bloco único, `e_ph` padrão com margem de confiança em vez da amostra bruta).

In [ ]:
import qkdpp
importlib.reload(qkdpp)

a_full = alice_bits.astype(np.uint8)
b_full = bob_bits.astype(np.uint8)

r = qkdpp.run(a_full, b_full, pe_fraction=PE_DISCLOSE, n_passes=10, seed=1)
print(r.summary())
print(f"e_ph usado (com margem de confiança): calculado internamente a partir da amostra de PE")

qkdpp.io.save_bits(work / "bob_final_qkdpp.txt", r.key)
print("chave final salva em", work / "bob_final_qkdpp.txt")

## 5. Resumo comparativo

**Atenção**: os comprimentos finais abaixo não são diretamente comparáveis como medida de segurança. O AIT dimensiona a PA pela taxa de erro *verdadeira pós-correção* (`nErrorRate`, com retrospecto); o `qkdpp` usa uma estimativa de PE *pré-registrada com margem de confiança* (mais conservador e mais rigoroso). Espera-se que o AIT produza uma chave maior por esse motivo -- isso não indica que o pipeline AIT seja "melhor", apenas que está usando um critério de dimensionamento menos rigoroso.

In [ ]:
ait_final_len = (work / "bob_final_ait.bin").stat().st_size * 8 if (work / "bob_final_ait.bin").exists() else 0
qkdpp_final_len = r.final_len

print(f"{'':20s} {'bits finais':>12s}")
print(f"{'AIT (com PA)':20s} {ait_final_len:>12d}")
print(f"{'qkdpp (com PA)':20s} {qkdpp_final_len:>12d}")
print()
print(f"QBER verdadeiro dos dados brutos: {qber_true:.4f}")
print(f"QBER estimado pelo qkdpp (amostra de PE): {r.qber:.4f}")